# TJTO - Tribunal de Justica do Estado do Tocantins

Scraper for the jurisprudence search of TJTO.

| Feature | Status |
|---------|--------|
| cjsg (Acordaos - 2nd instance) | Supported |
| cjsg (Decisoes Monocraticas - 2nd instance) | Supported |
| cjpg (Sentencas - 1st instance) | Supported |
| cjsg_ementa (ementa of a document by UUID) | Supported |
| cpopg | Not implemented |
| cposg | Not implemented |

The search portal sits behind an AWS WAF that blocks an IP after about 7 searches in a row and lets it through again after 10 minutes or more. Each page is one search, so this notebook asks for a single page per call and reuses each result in several cells. When the WAF blocks before any page, the call raises `BotChallengeBlockedError`; when it blocks in the middle of a pagination, it raises `ColetaParcialError`, with the pages already downloaded in `.resultado`.

## Basic search (2nd instance)

In [1]:
import juscraper as jus

tjto = jus.scraper('tjto')

dados_cjsg = tjto.cjsg('dano moral', paginas=1)
print(dados_cjsg.shape)
dados_cjsg.head(3)

Baixando páginas TJTO:   0%|          | 0/1 [00:00<?, ?it/s]

Baixando páginas TJTO: 100%|██████████| 1/1 [00:03<00:00,  3.31s/it]

Baixando páginas TJTO: 100%|██████████| 1/1 [00:03<00:00,  3.31s/it]

(20, 10)


,processo,uuid,classe,tipo_julgamento,assunto,competencia,relator,data_autuacao,data_julgamento,processo_link
0,0017731-25.2026.8.27.2700,f232618a47906695de76ea1489789d4b,Conflito de competência cível,Conflito de Competência,"Divisão e Demarcação, Propriedade, Coisas, DIR...",TRIBUNAL PLENO,ANGELA MARIA RIBEIRO PRUDENTE,2026-08-12,2026-10-15,https://eproc2.tjto.jus.br/consulta_publica/2G...
1,0014832-54.2026.8.27.2700,a33a5b8cdd8c3ff5f1a94fa64771c720,Agravo de Instrumento,Mérito,"Adicional de Insalubridade, Sistema Remunerató...",TURMAS DE DIREITO PÚBLICO,HÉLVIA TÚLIA SANDES PEDREIRA,2026-07-08,2026-10-07,https://eproc2.tjto.jus.br/consulta_publica/2G...
2,0025895-92.2025.8.27.2706,f46c6d2c10eea48862dc798c6f4d6d25,Apelação Cível,Mérito,"IPTU/ Imposto Predial e Territorial Urbano, Im...",TURMAS DE DIREITO PÚBLICO,HÉLVIA TÚLIA SANDES PEDREIRA,2026-08-19,2026-10-07,https://eproc2.tjto.jus.br/consulta_publica/2G...


## Available columns

In [2]:
dados_cjsg.columns.tolist()

['processo',
 'uuid',
 'classe',
 'tipo_julgamento',
 'assunto',
 'competencia',
 'relator',
 'data_autuacao',
 'data_julgamento',
 'processo_link']

## Preview result fields

In [3]:
row = dados_cjsg.iloc[0]
print(f"Processo: {row['processo']}")
print(f"Classe: {row['classe']}")
print(f"Relator: {row['relator']}")
print(f"Competencia: {row['competencia']}")
print(f"Data Julgamento: {row['data_julgamento']}")

Processo: 0017731-25.2026.8.27.2700
Classe: Conflito de competência cível
Relator: ANGELA MARIA RIBEIRO PRUDENTE
Competencia: TRIBUNAL PLENO
Data Julgamento: 2026-10-15


## Filters and document types

You can filter by judgment date range and choose the document type: `acordaos` (default), `decisoes` (monocratic decisions) or `sentencas` (sentences). The sort order is set with `ordenacao`: `"DESC"` (most recent first, the default), `"ASC"` or `"RELEV"`.

In [4]:
dados_decisoes = tjto.cjsg(
    'direito',
    paginas=1,
    tipo_documento='decisoes',
    data_julgamento_inicio='01/01/2025',
    data_julgamento_fim='30/06/2025',
)
print(dados_decisoes.shape)
dados_decisoes[['processo', 'classe', 'data_julgamento']].head(3)

Baixando páginas TJTO:   0%|          | 0/1 [00:00<?, ?it/s]

Baixando páginas TJTO: 100%|██████████| 1/1 [00:00<00:00,  1.12it/s]

Baixando páginas TJTO: 100%|██████████| 1/1 [00:00<00:00,  1.12it/s]

(20, 10)


,processo,classe,data_julgamento
0,0009807-94.2025.8.27.2700,Agravo de Instrumento,2025-06-30
1,0010075-51.2025.8.27.2700,Agravo de Instrumento,2025-06-30
2,0009926-55.2025.8.27.2700,Agravo de Instrumento,2025-06-30


## First instance search (cjpg), downloading and parsing separately

Use `cjpg()` to search first-instance jurisprudence; it accepts the same parameters as `cjsg()`. Both are a download followed by a parse, and the two steps are also available on their own: `cjpg_download()` returns the raw HTML pages and `cjpg_parse()` turns them into the same DataFrame (`cjsg_download()` and `cjsg_parse()` do the same for the 2nd instance).

In [5]:
brutos = tjto.cjpg_download('direito', paginas=1, tipo_documento='sentencas')
print(f"Pages downloaded: {len(brutos)}")
print(f"Raw HTML size (page 1): {len(brutos[0])} chars")

dados_cjpg = tjto.cjpg_parse(brutos)
print(f"\nParsed shape: {dados_cjpg.shape}")
dados_cjpg.head(3)

Baixando páginas TJTO:   0%|          | 0/1 [00:00<?, ?it/s]

Baixando páginas TJTO: 100%|██████████| 1/1 [00:01<00:00,  1.18s/it]

Baixando páginas TJTO: 100%|██████████| 1/1 [00:01<00:00,  1.18s/it]

Pages downloaded: 1
Raw HTML size (page 1): 1537222 chars



Parsed shape: (20, 10)


,processo,uuid,classe,tipo_julgamento,assunto,competencia,relator,data_autuacao,data_julgamento,processo_link
0,0017776-73.2025.8.27.2729,8c51da6b15e41f86a783d90a16a5e860,Procedimento do Juizado Especial Cível,Julgamento - Com Resolução do Mérito - Procedê...,"Perdas e Danos, Inadimplemento, Obrigações, DI...",JUIZADOS ESPECIAIS CÍVEIS,,2025-04-25,2026-10-20,https://eproc1.tjto.jus.br/consulta_publica/1G...
1,0016854-38.2024.8.27.2706,1ae5c8914a83a163813ddaeae128abe7,Busca e Apreensão em Alienação Fiduciária,Julgamento - Sem Resolução de Mérito - Extinçã...,"Alienação Fiduciária, Espécies de Contratos, O...",CIVEL,,2024-08-21,2026-10-20,https://eproc1.tjto.jus.br/consulta_publica/1G...
2,0020557-06.2026.8.27.2706,18492fa1b69e8248548ec787efa6ecaa,Busca e Apreensão em Alienação Fiduciária,Julgamento - Sem Resolução de Mérito - Extinçã...,"Alienação Fiduciária, Espécies de Contratos, O...",CIVEL,,2026-09-11,2026-10-20,https://eproc1.tjto.jus.br/consulta_publica/1G...
